# Hafta 1 özeti — NYC TLC zone × saat talep tahmini

Bu notebook 1.1–1.6 görevlerinde **ne yaptığımızı, neden yaptığımızı ve sonuçları** tek yerde topluyor.

- **Gri kod blokları (markdown içinde):** Script'lerden alınmış parçalar. Okumak için, çalıştırılmaz.
- **Çalıştırılabilir hücreler:** Sadece `data/agg` ve `data/processed` dosyalarını okur. Hiçbir şey indirmez, silmez ya da değiştirmez. **Kernel → Restart & Run All** ile güvenle çalıştırabilirsin.

| Görev | Konu | Durum |
|---|---|---|
| 1.1 | Repo ve klasör yapısı | ✅ |
| 1.2 | Veri sözlüğü, kolon seçimi | ✅ |
| 1.3 | Tek ay agregasyonu (DuckDB) | ✅ |
| 1.4 | Aylık ingest script'i | ✅ |
| 1.5 | 32 ayın çekilmesi | ✅ |
| 1.6 | Tam ızgara, sıfır doldurma, DST | ✅ |
| 1.7 | EDA | ⏳ sırada |

## 0. Kurulum

Notebook `notebooks/` klasöründen ya da proje kökünden açılabilir. Aşağıdaki hücre proje kökünü kendisi buluyor.

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
AGG = (ROOT / "data" / "agg" / "*.parquet").as_posix()
TARGET = (ROOT / "data" / "processed" / "target.parquet").as_posix()

con = duckdb.connect()
q = lambda sql: con.execute(sql).df()   # SQL çalıştır, sonucu pandas tablosu olarak al

print("Proje kökü:", ROOT)

## 1. Problem ve temel kararlar

**Soru:** Her taxi zone × her saat için talebi **bir gün önceden** tahmin etmek. D gününün sonunda D+1'in 24 saatini tahmin ediyoruz.

| Karar | Seçim | Neden |
|---|---|---|
| Veri seti | HVFHV (Uber/Lyft) | Hacmin büyük kısmı burada, seriler dolu. Yellow/Green/"For-Hire Vehicle" kapsam dışı. |
| Hedef zaman kolonu | `request_datetime` | Talebi, yani müşterinin aracı istediği anı ölçer. `pickup_datetime` ise karşılanan talebi gösterir ve arz kısıtına bağlıdır. 1.3'te %100 dolu olduğunu doğruladık. |
| Dönem | 2024-01 → 2026-08 (32 ay) | 5 Ocak 2025 congestion pricing kırılmasının öncesini ve sonrasını kapsıyor. |
| Kilitli holdout | 2026-07, 2026-08 | Sadece en sonda bir kez skorlanacak. EDA'da da kullanılmayacak. |
| Hariç tutulan zone'lar | 264, 265 | "Unknown" ve "Outside of NYC". Gerçek bir bölge değil. |
| Ufuk | 24 saat | Bu yüzden tüm feature'lar en az 24 saat geriden gelmek zorunda (2. hafta). |

## 2. Proje yapısı (Görev 1.1)

```
C:\zone_time_project\            ← OneDrive dışında (senkronizasyon ve kota sorunu olmasın diye)
├── src\
│   ├── ingest.py                ← indir → özetle → ham dosyayı sil
│   └── build_target.py          ← tam ızgara + sıfır + DST bayrakları
├── tests\
│   └── test_grid.py             ← hedef tablonun 6 kontrolü
├── data\
│   ├── raw\                     ← geçici; .gitignore'da
│   ├── agg\                     ← 32 aylık özet (repoda, ~16,5 MB)
│   └── processed\               ← target.parquet; .gitignore'da (her an yeniden üretilebilir)
├── notebooks\  sql\  reports\
├── config.yaml  requirements.txt  README.md  .gitignore
```

**Kural:** Tekrar çalışacak kod `src\` içinde `.py` dosyası olarak durur (4. haftada GitHub Actions bunları çalıştıracak). Keşif ve grafik işleri notebook'ta yapılır.

`.gitignore` içeriği ve sebepleri:

```
data/raw/            # ham parquet ayda ~0,5 GB; repoya girmez
*.part               # yarıda kalmış indirmeler
.ipynb_checkpoints/
__pycache__/
*.xlsx               # plan Excel'i kişisel notlar içeriyor, repo public
~$*                  # Excel'in açıkken oluşturduğu kilit dosyaları
*.pdf                # TLC veri sözlüğü; bizim dosyamız değil
data/processed/      # agg'den saniyeler içinde yeniden üretilir
```

## 3. Ham veriyi tanımak ve tek ay agregasyonu (Görev 1.2 – 1.3)

Ham dosya artık silindi, o yüzden bu bölümdeki kod çalıştırılmaz; sadece ne yaptığımızı gösteriyor.

**DuckDB neden?** Parquet kolon bazlı bir format. DuckDB sorguda geçen kolonları okur, dosyanın tamamını RAM'e yüklemez. 20 milyon satırlık bir ay birkaç saniyede özetlenir. pandas ile aynı iş birkaç GB RAM ister.

```python
f = r"C:\zone_time_project\fhvhv_tripdata_2024-01.parquet"   # r"..." → Windows'taki \ karakterleri sorun çıkarmasın

duckdb.sql(f"DESCRIBE SELECT * FROM '{f}'")     # kolonlar ve tipleri
duckdb.sql(f"SELECT count(*) FROM '{f}'")       # satır sayısı

# Zone × saat agregasyonu
duckdb.sql(f'''
    SELECT PULocationID AS zone,
           date_trunc('hour', request_datetime) AS hour,   -- 08:47:13 → 08:00:00
           count(*) AS trips
    FROM '{f}'
    WHERE request_datetime >= '2024-01-01'
      AND request_datetime <  '2024-02-01'                 -- ay dışına taşan kayıtları at
      AND PULocationID NOT IN (264, 265)
    GROUP BY zone, hour
''')
```

**Sonuçlar (Ocak 2024):**

| Kontrol | Değer | Yorum |
|---|---|---|
| Ham satır | 19.663.930 | Beklenen aralıkta (~20M) |
| `request_datetime` dolu | 19.663.930 (%100) | Hedef kolon güvenle kullanılabilir |
| Özet satır | 186.342 | Tam ızgara 263 × 744 = 195.672 olurdu; ~%4,8 zone-saat hiç yolculuk içermiyor → 1.6'da 0 ile doldurduk |
| Filtrelenen satır | 1.626 (%0,008) | İhmal edilebilir |
| En yoğun saatler | Zone 79 ve 148, Cuma/Cumartesi 23:00–01:00 | Talep tepesi gece hayatından geliyor (havalimanlarından değil) |

## 4. Aylık ingest script'i (Görev 1.4 – 1.5)

`src/ingest.py` her ay için aynı döngüyü uygular:

```
indir (.part olarak) ──► yeniden adlandır ──► DuckDB ile özetle ──► data/agg/YYYY-MM.parquet ──► ham dosyayı sil
```

Önemli parçalar:

```python
# Ay ay dolaşma: 2024-11 → 2024-12 → 2025-01 ...
def month_range(start, end): ...

# İdempotent: özet dosyası zaten varsa ayı atla
if os.path.exists(out) and not a.force:
    print("agregat zaten var, atlandi"); continue

# Yarıda kalan indirme gerçek dosya sanılmasın diye önce .part uzantısıyla iniyor
urllib.request.urlretrieve(URL.format(ym=ym), tmp)
os.replace(tmp, path)

# Bağlantıyı kapat → Windows dosya kilidini bırakır → silme hata vermez
con.close()
os.remove(raw)

# Güvenlik kontrolü: %1'den fazla satır filtrelendiyse uyar
if dropped / raw_n > 0.01: print("UYARI ...")
```

**Neden idempotent olması önemli?** Her ay kendi dosyasına yazılıyor. Aynı ay iki kez işlenirse üzerine yazılır ya da atlanır, asla iki kez eklenmez. Tek bir dosyaya ekleme (append) yapsaydık, iki kez işlenen bir ayın sayıları sessizce iki katına çıkardı.

**Çalıştırma:**
```
python src\ingest.py --start 2024-02 --end 2024-02     # tek ay testi
python src\ingest.py --start 2024-01 --end 2026-08     # tüm dönem (2024-01 atlandı → idempotent olduğu kanıtlandı)
```

**Sonuç:** 32 ay, hiçbir ayda filtrelenen oran %0,023'ü geçmedi. Aşağıdaki hücre bunu senin dosyalarından doğrular.

In [ ]:
files = sorted((ROOT / "data" / "agg").glob("*.parquet"))
print(f"Özet dosya sayısı: {len(files)}  |  toplam boyut: {sum(f.stat().st_size for f in files)/1e6:.1f} MB")

monthly = q(f'''
    SELECT strftime(hour, '%Y-%m') AS ay,
           count(*)               AS ozet_satir,
           count(DISTINCT zone)   AS zone_sayisi,
           sum(trips)             AS yolculuk
    FROM read_parquet('{AGG}')
    GROUP BY ay ORDER BY ay
''')
monthly

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.5))
months = list(monthly["ay"])
x = range(len(months))
ax.plot(x, monthly["yolculuk"] / 1e6, marker="o")
if "2025-01" in months:
    i = months.index("2025-01")
    ax.axvline(i, color="grey", ls="--", lw=1)
    ax.text(i, ax.get_ylim()[1], " congestion pricing", va="top", fontsize=9, color="grey")
hold = [i for i, m in enumerate(months) if m in ("2026-07", "2026-08")]
if hold:
    ax.axvspan(min(hold) - 0.5, max(hold) + 0.5, color="orange", alpha=0.15, label="kilitli holdout")
    ax.legend(loc="lower right")
ax.set_xticks(list(x), months, rotation=90)
ax.set_ylabel("Aylık yolculuk (milyon)")
ax.set_title("Aylık HVFHV talebi (request bazlı, 264/265 hariç)")
plt.tight_layout()
plt.show()

## 5. Hedef tablo: tam ızgara, sıfır doldurma, DST (Görev 1.6)

`src/build_target.py` üç iş yapıyor.

**a) Tam ızgara + sıfır doldurma.** `GROUP BY` sadece yolculuk olan saatleri üretir. Izgarayı tamamlamazsak model sıfır talebi hiç görmez ve `lag_24` "dün aynı saat" yerine "son dolu saat" olur.

```sql
WITH zones AS (SELECT DISTINCT zone FROM agg),
     hrs   AS (SELECT unnest(range(min_saat, max_saat + 1 saat, 1 saat)) AS hour),
     grid  AS (SELECT * FROM zones CROSS JOIN hrs)          -- her zone × her saat
SELECT g.zone, g.hour, coalesce(a.trips, 0) AS trips        -- eşleşme yoksa 0
FROM grid g LEFT JOIN agg a USING (zone, hour)
```

**b) DST (yaz saati) bayrakları.** Veri timezone'suz, yerel NYC saatiyle tutuluyor.

| Durum | Ne oluyor | Ne yaptık |
|---|---|---|
| Mart, ileri alma (02:00) | Bu saat yaşanmıyor | `trips = NULL`, `is_dst_gap = True`. 0 değil, çünkü "talep yok" değil "saat yok". Model bu satırları kullanmayacak. |
| Kasım, geri alma (01:00) | İki gerçek saat tek satıra düşüyor, sayı ~2 kat | Satır kalıyor, `is_dst_dup = True`, `trips_model = trips / 2` |

Tabloda iki sayı kolonu var: `trips` ham sayı (dokunulmuyor), `trips_model` ise model için düzeltilmiş kolon. Feature'lar ve hedef `trips_model`'den üretilecek. Böylece Kasım'daki ~2 kat değer `lag_24`, `lag_168` ve rolling ortalamalar üzerinden sonraki günlere yayılmıyor.

DST saatleri Python'un `America/New_York` saat dilimi bilgisinden otomatik bulunuyor, elle yazılmadı.

**c) Kontrol raporu ve testler.** Aşağıdaki hücreler aynı kontrolleri senin `target.parquet` dosyan üzerinde tekrar çalıştırıyor.

In [ ]:
summary = q(f'''
    SELECT count(DISTINCT zone)                                  AS zone_sayisi,
           count(DISTINCT hour)                                  AS saat_sayisi,
           count(*)                                              AS satir,
           count(DISTINCT zone) * count(DISTINCT hour)           AS beklenen_satir,
           sum(trips)                                            AS toplam_yolculuk,
           round(avg((trips = 0)::INT) FILTER (WHERE NOT is_dst_gap), 4) AS sifir_orani,
           sum(is_dst_gap::INT)                                  AS dst_bosluk_satir,
           sum(is_dst_dup::INT)                                  AS dst_cift_satir
    FROM read_parquet('{TARGET}')
''')
summary.astype(object).iloc[0].to_frame("değer")

In [ ]:
# DST doğrulaması: geçiş saatini bir önceki ve bir sonraki saatle karşılaştır (tüm zone'lar toplamı)
dst = q(f'''
    WITH tot AS (SELECT hour, sum(trips) AS t FROM read_parquet('{AGG}') GROUP BY hour),
         flags AS (SELECT DISTINCT hour, is_dst_gap, is_dst_dup FROM read_parquet('{TARGET}')
                   WHERE is_dst_gap OR is_dst_dup)
    SELECT f.hour AS gecis_saati,
           CASE WHEN f.is_dst_gap THEN 'ileri (boşluk)' ELSE 'geri (çift)' END AS tur,
           (SELECT t FROM tot WHERE hour = f.hour - INTERVAL 1 HOUR) AS once,
           (SELECT t FROM tot WHERE hour = f.hour)                   AS o_saat,
           (SELECT t FROM tot WHERE hour = f.hour + INTERVAL 1 HOUR) AS sonra
    FROM flags f ORDER BY f.hour
''')
dst["o_saat / komşu ort."] = (dst["o_saat"] / ((dst["once"] + dst["sonra"]) / 2)).round(2)
dst

**Okuma:** "İleri" satırlarında oran ~0 (saat yaşanmıyor), "geri" satırlarında ~2 (iki saat tek satırda). Bu sonuç, verinin yerel saatle tutulduğu varsayımını doğruluyor.

In [ ]:
# En düşük hacimli zone'lar — 1.7'de (EDA) çıkarılıp çıkarılmayacağına karar verilecek
low = q(f'''
    SELECT zone,
           sum(trips)                         AS toplam,
           round(avg((trips = 0)::INT), 3)    AS sifir_orani,
           round(100.0 * sum(trips) / (SELECT sum(trips) FROM read_parquet('{TARGET}')), 5) AS hacim_payi_yuzde
    FROM read_parquet('{TARGET}')
    WHERE NOT is_dst_gap
    GROUP BY zone ORDER BY toplam LIMIT 10
''')
lookup = ROOT / "taxi_zone_lookup.csv"
if lookup.exists():
    z = pd.read_csv(lookup).rename(columns={"LocationID": "zone"})
    low = low.merge(z[["zone", "Borough", "Zone"]], on="zone", how="left")
else:
    print("Not: taxi_zone_lookup.csv proje kökünde yok; zone isimleri eklenmedi.")
low

### Testler (`tests/test_grid.py`)

| Test | Neyi yakalar |
|---|---|
| `test_grid_is_complete` | Satır sayısı = zone × saat değilse ızgara eksiktir |
| `test_hours_are_contiguous` | Arada atlanmış bir saat var mı |
| `test_no_duplicate_zone_hour` | Bir ay iki kez yazıldıysa burada patlar |
| `test_excluded_zones_absent` | 264/265 sızmış mı |
| `test_no_negative_and_null_only_on_dst_gap` | NULL sadece DST boşluk saatinde olmalı |
| `test_trips_model_dst_rules` | `trips_model` boşlukta NULL, çift saatte trips/2, diğer saatlerde trips mi |
| `test_trip_total_preserved` | Toplam yolculuk sayısı agg ile aynı mı (DST boşluğu hariç) |

Çalıştırma: `python -m pytest tests -v` → **7 passed**.

Testleri burada sahte veriyle de denedim: bir satırı bilerek sildiğimde `test_grid_is_complete` hatayı yakaladı.

## 6. Yol boyunca karşılaşılan sorunlar

| Sorun | Sebep | Çözüm |
|---|---|---|
| `SyntaxError: (unicode error) 'unicodeescape'` | Windows yolundaki `\U` (`C:\Users`) Python'da özel karakter kodu sanıldı | `r"C:\..."` (raw string) ya da `/` kullan |
| `IO Error: No files found` | `C:Users` yazılmıştı, `C:` sonrasında `\` eksikti | `os.path.exists(f)` ile önce kontrol et |
| OneDrive senkronizasyonu | Masaüstü OneDrive'a bağlı; 32 × 0,5 GB kotayı doldururdu ve dosyaları kilitleyebilirdi | Proje `C:\zone_time_project` konumuna taşındı |
| `git add ... unknown switch 'm'` | Üç git komutu tek satıra yapıştırılmıştı | Komutları tek tek çalıştır |
| `.gitignore` değişikliği GitHub'a gitmedi | Dosya değiştirildi ama commit'lenmedi | `git add .gitignore` → commit → push; `git status` = "working tree clean" |

## 7. Açık kararlar → Görev 1.7 (EDA)

1. **Düşük hacimli zone'lar:** Sıfır oranı %90'ın üstünde olan 5 zone'u (1, 105, 199, 110, 2) çıkarmayı öneriyorum. Toplam hacmin %0,001'inden azını taşıyorlar.
2. **EDA kapsamı:** Saatlik ve haftalık sezonsallık, zone Pareto dağılımı, Ocak 2025 kırılması, havalimanları (1, 132, 138). EDA'ya holdout ayları (2026-07, 2026-08) dahil edilmeyecek.